In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Transfomer Decoder:
# 1. token embedding
# 2. position embedding
# 3. Multihead attention
# 4. Feed forward Nural network
# 5. attention +FFN ：decoder block
# 6. 多个decoder堆叠起来
# 7.laynorm、activation、残差连接

In [3]:
class FeedForwardNeuralNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super(FeedForwardNeuralNetwork, self).__init__()
        # Layer normalization
        self.layer_norm = nn.LayerNorm(d_model)
        # Linear projection layers
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        # Activation function
        self.activation = nn.GELU()
        
    def forward(self, x):
        """
        Forward pass of the feed-forward block.

        Args:
            x (Tensor): Input tensor of shape [batch_size, seq_len, hidden_size].

        Returns:
            Tensor: Output tensor of the same shape as input.
        """
        residual = x
        output = self.layer_norm(x)                 # [batch_size, seq_len, hidden_size]
        output = self.linear1(output)               # [batch_size, seq_len, hidden_size * 4]
        output = self.activation(output)            # [batch_size, seq_len, hidden_size * 4]
        output = self.linear2(output)               # [batch_size, seq_len, hidden_size]
        
        return residual + output


In [4]:
batch_size, seq_len, hidden_size = 16, 10, 768

x = torch.randn(batch_size, seq_len, hidden_size)

ffn = FeedForwardNeuralNetwork(768, 768*4)

output = ffn(x)

print(f"x size is {x.size()}")
print(f"output size is {output.size()}")
print(f"output is {output[0]}")

x size is torch.Size([16, 10, 768])
output size is torch.Size([16, 10, 768])
output is tensor([[-1.0940,  0.0967,  1.1898,  ..., -0.9256,  1.0180,  1.7866],
        [ 0.1402,  1.2957, -0.9932,  ...,  1.3418, -0.0288, -1.1619],
        [-0.1466,  1.6384, -0.2307,  ...,  0.8089,  1.6256,  0.1903],
        ...,
        [ 0.8485, -2.4715, -2.5040,  ..., -0.0732, -0.1672,  1.3837],
        [-0.4466,  1.2567, -0.4782,  ...,  2.0054, -0.7160, -1.3726],
        [-0.5499,  1.1868,  0.1414,  ...,  0.1603, -1.0661,  2.6898]],
       grad_fn=<SelectBackward0>)


In [5]:
class ScaledDotProductAttention(nn.Module):
    """Scaled Dot-Product Attention
    
    Computes the attention weights using the formula:
        Attention(Q, K, V) = softmax((Q * K^T) / sqrt(d_model))
    """
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, query, key, value, mask=None):
        """
        Compute attention weights and output.

        Args:
            query: Query tensor of shape [batch_size, seq_len, d_model]
            key: Key tensor of shape [batch_size, seq_len, d_model]
            value: Value tensor of shape [batch_size, seq_len, d_model]
            mask: Optional mask tensor (same shape as attention scores)

        Returns:
            output: Attention output tensor [batch_size, seq_len, d_model]
            attention_weights: Attention weights [batch_size, seq_len, seq_len]
        """
        d_q = query.size()[-1]

        # Compute scaled dot-product attention scores - [batch_size, seq_len, seq_len]
        scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_q, dtype=torch.float32))

        # Apply mask (if provided)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # Apply softmax to get attention weights
        attention_weights = torch.softmax(scores, dim=-1)

        # Compute the final output  - - [batch_size, seq_len, d_model]
        output = torch.matmul(attention_weights, value)

        return output, attention_weights

In [6]:
class MultiheadAttention(nn.Module):
    """
    Multi-Head Attention Module

    Splits the input into multiple heads, performs Scaled Dot-Product Attention 
    on each head independently, and then concatenates the results followed by a 
    linear projection.
    """
    
    def __init__(self, d_model, num_heads):
        """
        Initialize the Multi-Head Attention layer.

        Args:
            d_model (int): The dimensionality of the model.
            num_heads (int): The number of attention heads.
        """
        super(MultiheadAttention, self).__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        
        # Ensure the model dimension is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.head_dim = self.d_model // self.num_heads
        
        # Linear projections for query, key, and value
        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.val_proj = nn.Linear(d_model, d_model)
        
        # Output projection after concatenation
        self.out_proj = nn.Linear(d_model, d_model)
        
        # Scaled Dot-Product Attention module
        self.attention = ScaledDotProductAttention()
    
    def split_heads(self, x):
        """
        Split the last dimension into (num_heads, head_dim) 
        and transpose to shape [batch_size, num_heads, seq_len, head_dim].

        Args:
            x: Tensor of shape [batch_size, seq_len, d_model]

        Returns:
            Tensor of shape [batch_size, num_heads, seq_len, head_dim]
        """
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
    
    def combine_heads(self, x):
        """
        Combine multiple heads into a single tensor.

        Args:
            x: Tensor of shape [batch_size, num_heads, seq_len, head_dim]

        Returns:
            Tensor of shape [batch_size, seq_len, d_model]
        """
        batch_size, num_heads, seq_len, head_dim = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, num_heads * head_dim)
        
    def forward(self, x, mask=None):
        """
        Perform multi-head attention computation.

        Args:
            x: Input tensor [batch_size, seq_len, d_model]
            mask: Optional mask tensor [batch_size, seq_len]

        Returns:
            output: Final output tensor [batch_size, seq_len, d_model]
            attention_weights: Attention weights [batch_size, num_heads, seq_len, seq_len]
        """
        batch_size, seq_len, d_model = x.size()
        
        # Linear projections
        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.val_proj(x)
        
        # Split into multiple heads
        query_split = self.split_heads(query)
        key_split = self.split_heads(key)
        value_split = self.split_heads(value)
        
        # Expand mask for all heads
        if mask is not None:
            mask = mask.unsqueeze(1)
        
        # Compute attention
        attention_out, attention_weights = self.attention(query_split, key_split, value_split, mask)
        # attention_out shape: [batch_size, num_heads, seq_len, head_dim]

        # Combine heads
        attention_out = self.combine_heads(attention_out)
        
        # Final linear projection
        output = self.out_proj(attention_out)
        
        return output, attention_weights

In [51]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super(TransformerDecoderBlock, self).__init__()
        # Core modules
        self.attention = MultiheadAttention(d_model, num_heads)
        self.feedForward = FeedForwardNeuralNetwork(d_model, d_ff)
        # Final layer normalization
        self.layer_norm = nn.LayerNorm(d_model)
        
    def forward(self, x, attn_mask=None):
        """
        Forward pass for a single Transformer decoder block.

        Args:
            x (Tensor): Input tensor of shape [batch_size, seq_len, d_model].
            attn_mask (Tensor, optional): Attention mask to prevent attending to certain positions 
                                          (e.g., future tokens during autoregressive decoding).

        Returns:
            Tuple[Tensor, Tensor]:
                - output: The processed tensor after attention, feed-forward network, and normalization.
                - attn_weights: Attention weights from the multi-head attention layer.
        """
        # Multi-head self-attention
        attn_output, attn_weights = self.attention(x, attn_mask)

        # Feed-forward network with residual connection
        ff_output = self.feedForward(x + attn_output)

        # Apply final layer normalization
        output = self.layer_norm(ff_output)
        
        return output, attn_weights


In [52]:
batch_size, seq_len, hidden_size = 16, 10, 768

x = torch.randn(batch_size, seq_len, hidden_size)

tdb = TransformerDecoderBlock(hidden_size, 12, hidden_size*4)

output, attn_weights = tdb(x)
print(f"x size is {x.size()}")
print(f"output size is {output.size()}")
print(f"output is {output[0]}")

x size is torch.Size([16, 10, 768])
output size is torch.Size([16, 10, 768])
output is tensor([[ 1.1633, -0.8814, -1.4511,  ...,  0.5813, -1.6208, -0.7332],
        [-1.0043, -1.2173, -0.9074,  ...,  0.5185, -0.4711,  0.8636],
        [-0.2395,  0.5426, -0.5056,  ..., -1.4466,  0.8692,  0.7440],
        ...,
        [-0.3256, -0.8025,  0.4692,  ...,  0.1958,  0.2536,  0.4729],
        [-1.5510,  0.2248, -0.8158,  ...,  0.1999,  0.3464,  0.8379],
        [-0.7206,  0.0965, -0.2437,  ..., -0.7385, -0.8052, -2.1816]],
       grad_fn=<SelectBackward0>)


In [53]:
class PositionwiseFeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network (with GeLU activation)

    Applies two linear transformations with a GeLU activation in between,
    independently to each position in the sequence. Includes Layer Normalization 
    and residual connections for stability.
    """
    def __init__(self, d_model, d_ff, dropout_rate=0.1):
        """
        Initialize the feed-forward network.

        Args:
            d_model (int): Dimensionality of the input and output.
            d_ff (int): Dimensionality of the inner feed-forward layer.
            dropout_rate (float): Dropout probability (currently optional).
        """
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.activation = nn.GELU()  # GeLU activation function
        # self.res_dropout = nn.Dropout(dropout_rate)  # Optional dropout

    def forward(self, x):
        """
        Forward pass of the position-wise feed-forward layer.

        Args:
            x: Input tensor [batch_size, seq_len, d_model]

        Returns:
            Tensor of shape [batch_size, seq_len, d_model] after applying 
            feed-forward transformation and residual connection.
        """
        residual = x  # Residual connection
        output = self.layer_norm(x)  # Pre-layer normalization
        output = self.linear1(output)
        output = self.activation(output)
        output = self.linear2(output)
        
        # Optionally apply dropout before adding the residual
        # output = self.res_dropout(output)
        
        return output + residual

In [54]:
class TransformerDecoderBlock(nn.Module):
    """
    Transformer Decoder Block (pure decoder structure)

    A single block of the Transformer decoder that includes:
    - Masked Multi-Head Self-Attention
    - Position-wise Feed-Forward Network
    - Layer Normalization and residual connections
    """
    def __init__(self, d_model, num_heads, d_ff):
        """
        Initialize the Transformer decoder block.

        Args:
            d_model (int): Dimensionality of the model embeddings.
            num_heads (int): Number of attention heads.
            d_ff (int): Dimensionality of the feed-forward network.
        """
        super().__init__()
        
        # Core modules
        self.self_attn = MultiheadAttention(d_model, num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff)
        
        # Final normalization after feed-forward
        self.final_norm = nn.LayerNorm(d_model)
        
        # Optional dropout for additional regularization
        # self.block_dropout = nn.Dropout(dropout_rate)

    def forward(self, x, attn_mask=None):
        """
        Forward pass for the Transformer decoder block.

        Args:
            x: Input tensor [batch_size, seq_len, d_model]
            attn_mask: Optional attention mask tensor

        Returns:
            output: Output tensor [batch_size, seq_len, d_model]
            attn_weights: Attention weights 
                          [batch_size, num_heads, seq_len, seq_len]
        """
        # Self-attention layer (with optional causal mask)
        attn_output, attn_weights = self.self_attn(x, attn_mask)
        
        # Position-wise feed-forward layer
        ff_output = self.feed_forward(attn_output)
        
        # Final normalization
        output = self.final_norm(ff_output)
        
        # Optional dropout
        # output = self.block_dropout(output)
        
        return output, attn_weights


In [55]:
model = TransformerDecoder(
    vocab_size=500, 
    d_model=256, 
    max_len=128, 
    num_layers=12, 
    num_heads=8, 
    d_ff=256*4)


AttributeError: 'TransformerDecoderBlock' object has no attribute 'attention'

In [36]:
batch_size, seq_len = 16,10

input_ids = torch.randint(0,500,(batch_size, seq_len))

print(f"input size is {input_ids.size()}")
print(f"input is {input_ids}")

output, _ = model(input_ids)

print(f"output size is {output.size()}")
print(f"output is {output[-1]}")


input size is torch.Size([16, 10])
input is tensor([[282,  60, 282, 303, 283, 334, 174,  34,  43, 119],
        [169, 368, 372, 160, 486, 166, 143, 347,  76, 336],
        [200, 360,  60,  40, 491, 210, 353, 320, 188, 333],
        [395, 305, 201, 302, 498, 104, 152, 244, 395, 275],
        [489, 346, 103, 411,  41, 475, 229, 471,  89, 155],
        [322, 113,  19, 183, 286, 304,  37, 439, 227, 127],
        [433, 473, 143, 265, 445, 452, 163, 265,   0, 355],
        [391, 262, 209, 375, 450, 377, 154, 103, 224, 304],
        [ 20, 295,  71, 183, 156, 412,  99,  72, 188,  12],
        [306, 133,  36, 190, 350, 144, 138, 293, 468, 107],
        [175, 468,  71, 192, 384, 319, 425, 494, 257,  73],
        [116, 382, 413, 206, 340, 409, 387, 138, 481, 212],
        [125, 281, 404, 391, 173, 392, 338, 360,   7, 226],
        [321,  80, 325,  80, 471,  31,  50, 367, 410, 148],
        [204, 139, 202, 140, 311, 240, 375, 333, 401, 445],
        [174, 371, 451,  40,  97, 403, 468, 260, 381, 31